In [1]:
import json
import time

import joblib
import numpy as np
import pandas as pd
import torch

from benchmark.config import DATA_PROCESSED, DATASETS, MODELS_DIR, RESULTS_BENCHMARK
from benchmark.metrics import evaluate_regression
from benchmark.models.xrfm_model import make_processor
from benchmark.models.xrfm_model import fit_best_xrfm, fit_xrfm_cross_validation

data_name = "diamonds"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
train_df = pd.read_csv(processed / "train.csv")
train_df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z,volumn
0,1.26,Ideal,J,SI1,62.0,56.0,5301,6.90,6.93,4.29,205.13
1,1.04,Ideal,G,VS2,61.6,55.0,6617,6.51,6.55,4.02,171.41
2,0.72,Ideal,H,IF,62.1,58.0,3283,5.72,5.75,3.56,117.09
3,0.26,Very Good,E,VVS2,61.4,57.0,506,4.10,4.13,2.52,42.67
4,0.23,Very Good,E,VVS2,63.0,58.0,505,3.95,3.99,2.50,39.40


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43152 entries, 0 to 43151
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    43152 non-null  float64
 1   cut      43152 non-null  object 
 2   color    43152 non-null  object 
 3   clarity  43152 non-null  object 
 4   depth    43152 non-null  float64
 5   table    43152 non-null  float64
 6   price    43152 non-null  int64  
 7   x        43152 non-null  float64
 8   y        43152 non-null  float64
 9   z        43152 non-null  float64
 10  volumn   43152 non-null  float64
dtypes: float64(7), int64(1), object(3)
memory usage: 3.6+ MB


In [4]:
test_df = pd.read_csv(processed / "test.csv")
test_df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z,volumn
0,1.20,Very Good,I,VVS2,59.9,55.0,6558,6.88,6.94,4.14,197.67
1,0.30,Very Good,I,VS1,62.9,58.0,473,4.26,4.29,2.69,49.16
2,0.41,Premium,G,VVS2,61.6,56.0,1076,4.78,4.76,2.94,66.89
3,0.43,Ideal,E,VS2,62.2,57.0,981,4.80,4.84,3.00,69.70
4,0.53,Very Good,E,VVS2,62.3,55.0,2280,5.18,5.22,3.23,87.34


In [5]:
with open(processed / "metadata.json") as f:
    metadata = json.load(f)

print(metadata)

{'num_cols': ['carat', 'depth', 'table', 'x', 'y', 'z', 'volumn'], 'cat_cols': ['cut', 'color', 'clarity'], 'target_col': 'price'}


In [6]:
X = train_df.drop(columns=metadata["target_col"])
y = train_df[metadata["target_col"]].to_numpy().astype(np.float32)

processor = make_processor(metadata)
n_num_cols = len(metadata["num_cols"])

In [7]:
best_trial = fit_xrfm_cross_validation(
    X, y, processor, n_num_cols, task=cfg["task"], device=device
)

print(f"Best params: {best_trial.params}")
print(f"Best MSE: {best_trial.value}")

[I 2026-04-23 00:36:36,402] A new study created in memory with name: no-name-d7435424-b049-4c8d-a13b-c887621a9080


None
Fitting xRFM with 1 trees and 0 iterations per tree



Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Using top_vector_agop_on_subset split method


Getting AGOP on subset
X_train torch.Size([24617, 27]) y_train torch.Size([24617, 1]) X_val torch.Size([1296, 27]) y_val torch.Size([1296, 1])
Fitting RFM with ntrain: 24617, d: 27, and nval: 1296


...


Tuning split temperature: 100%|██████████| 36/36 [00:39<00:00,  1.27s/it]


Tuning split temperature: 100%|██████████| 36/36 [00:39<00:00,  1.09s/it]

Selected split_temperature=0.13414886285344563 based on validation mse=294898.343750
Using soft routing for tree prediction


[I 2026-04-23 06:34:24,363] Trial 99 finished with value: 284761.68125 and parameters: {'exponent': 1.3082530738165434, 'kernel_type': 'kpq', 'bandwidth': 1.2933379751208478, 'diag': False, 'norm_p': 1.824778631708211, 'reg': 0.015318863977567468}. Best is trial 86 with value: 272211.79375.


Best params: {'exponent': 1.3488314714785494, 'kernel_type': 'kpq', 'bandwidth': 3.392738013710273, 'diag': False, 'norm_p': 1.744017313867038, 'reg': 0.0137131961354413}
Best MSE: 272211.79375


In [8]:
model, pipeline, fit_time, train_metrics, val_metrics = fit_best_xrfm(
    X,
    y,
    best_trial,
    processor,
    n_num_cols,
    task=cfg["task"],
    device=device,
)

print(f"Training time: {fit_time:.2f}s")
print(f"Train RMSE: {train_metrics['rmse']:.4f}")
print(f"Val RMSE:   {val_metrics['rmse']:.4f}")

joblib.dump(model, MODELS_DIR / data_name / "xrfm.joblib")
joblib.dump(pipeline, MODELS_DIR / data_name / "xrfm_pipeline.joblib")

In [9]:
model = joblib.load(MODELS_DIR / data_name / "xrfm.joblib")
pipeline = joblib.load(MODELS_DIR / data_name / "xrfm_pipeline.joblib")

In [10]:
X_test = test_df.drop(columns=metadata["target_col"])
y_test = test_df[metadata["target_col"]].to_numpy().astype(np.float32)

X_test = pipeline.transform(X_test)
t0 = time.perf_counter()
y_pred = model.predict(X_test)
infer_time = (time.perf_counter() - t0) / len(X_test)

Using soft routing for tree prediction


In [11]:
results = evaluate_regression(y_test, y_pred)
print(f"RMSE: {results['rmse']:.4f}")
print(f"Inference time per sample: {infer_time * 1e6:.2f}µs")

RMSE: 520.0493
Inference time per sample: 110.26µs


In [12]:
output = {
    **results,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "fit_time_s": fit_time,
    "infer_time_per_sample_s": infer_time,
    "best_params": {
        "tuning_metric": model.tuning_metric,
        "split_method": model.split_method,
        "max_leaf_size": model.max_leaf_size,
        **model.rfm_params,
    },
}

results_dir = RESULTS_BENCHMARK / "diamonds"
with open(results_dir / "xrfm.json", "w") as f:
    json.dump(output, f, indent=4)

print(f"Results saved to {results_dir / 'xrfm.json'}")